## CAP6938 AI Knowledge TESTER

### Part 1: 

TEST

In [1]:
import sys
# initialization code
import numpy as np
from keras.datasets import mnist
%matplotlib inline
from matplotlib import pyplot as plt
from scipy import stats
import torch
import torch.nn as nn

print("Python:", sys.version)
print("Executable:", sys.executable)

packages = [
    "torch",
    "torchvision",
    "transformers",
    "datasets",
    "sentence_transformers",
    "PIL",
    "numpy",
    "matplotlib",
]

for package in packages:
    try:
        module = __import__(package)
        version = getattr(module, "__version__", "installed (version unavailable)")
        print(f"{package:22s} OK   {version}")
    except ImportError as e:
        print(f"{package:22s} MISSING   {e}")

Python: 3.10.4 (tags/v3.10.4:9d38120, Mar 23 2022, 23:13:41) [MSC v.1929 64 bit (AMD64)]
Executable: C:\Python310\python.exe
torch                  OK   2.5.1+cu121
torchvision            OK   0.20.1+cu121
transformers           OK   5.17.0
datasets               OK   4.8.4
sentence_transformers  OK   6.0.1
PIL                    OK   12.2.0
numpy                  OK   1.26.4
matplotlib             OK   3.7.2


In [2]:
# Sets device to "cuda" if a GPU is available  (in Colabs, enable GPU by Edit->Notebook Settings-->Hardware Accelerator=GPU)
device = "cuda" if torch.cuda.is_available() else 'cpu'
print(device) # make sure you're using GPU instance
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version used by torch:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

cuda
Torch version: 2.5.1+cu121
CUDA available: True
CUDA version used by torch: 12.1
GPU: NVIDIA GeForce RTX 2070


#### 1a


In [3]:
try:
    import faiss
    print("faiss                  OK  ", faiss.__version__)
except ImportError as e:
    print("faiss                  MISSING  ", e)

faiss                  OK   1.15.1


In [4]:
import json
from pathlib import Path
from pprint import pprint

DATA_DIR = Path("../Yelp_JSON")
BUSINESS_FILE = DATA_DIR / "yelp_academic_dataset_business.json"

with BUSINESS_FILE.open("r", encoding="utf-8") as f:
    first_business = json.loads(f.readline())

pprint(first_business)

{'address': '1616 Chapala St, Ste 2',
 'attributes': {'ByAppointmentOnly': 'True'},
 'business_id': 'Pns2l4eNsfO8kk83dixA6A',
 'categories': 'Doctors, Traditional Chinese Medicine, Naturopathic/Holistic, '
               'Acupuncture, Health & Medical, Nutritionists',
 'city': 'Santa Barbara',
 'hours': None,
 'is_open': 0,
 'latitude': 34.4266787,
 'longitude': -119.7111968,
 'name': 'Abby Rappoport, LAC, CMQ',
 'postal_code': '93101',
 'review_count': 7,
 'stars': 5.0,
 'state': 'CA'}


In [5]:
import json

restaurant_count = 0
total_businesses = 0
restaurant_cities = {}

with BUSINESS_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        business = json.loads(line)
        total_businesses += 1

        categories = business.get("categories") or ""
        category_list = [
            category.strip().lower()
            for category in categories.split(",")
        ]

        if "restaurants" in category_list:
            restaurant_count += 1

            city = business.get("city", "Unknown")
            restaurant_cities[city] = restaurant_cities.get(city, 0) + 1

print(f"Total businesses: {total_businesses:,}")
print(f"Restaurants: {restaurant_count:,}")

print("\nTop 10 cities by restaurant count:")
for city, count in sorted(
    restaurant_cities.items(),
    key=lambda item: item[1],
    reverse=True
)[:10]:
    print(f"{city}: {count:,}")

Total businesses: 150,346
Restaurants: 52,268

Top 10 cities by restaurant count:
Philadelphia: 5,852
Tampa: 2,960
Indianapolis: 2,862
Nashville: 2,502
Tucson: 2,466
New Orleans: 2,259
Edmonton: 2,166
Saint Louis: 1,790
Reno: 1,286
Boise: 847


In [6]:
import json
from pprint import pprint

REVIEW_FILE = DATA_DIR / "yelp_academic_dataset_review.json"

# Step 1: Select a Tampa restaurant with at least 100 reviews.
selected_restaurant = None

with BUSINESS_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        business = json.loads(line)

        categories = [
            c.strip().lower()
            for c in (business.get("categories") or "").split(",")
        ]

        if (
            business.get("city") == "Tampa"
            and "restaurants" in categories
            and business.get("review_count", 0) >= 100
        ):
            selected_restaurant = business
            break

if selected_restaurant is None:
    raise ValueError("No matching Tampa restaurant found.")

print("SELECTED RESTAURANT")
print(f"Name: {selected_restaurant['name']}")
print(f"Business ID: {selected_restaurant['business_id']}")
print(f"Categories: {selected_restaurant['categories']}")
print(f"Rating: {selected_restaurant['stars']}")
print(f"Review count: {selected_restaurant['review_count']}")

# Step 2: Retrieve up to five reviews for this restaurant.
reviews = []
business_id = selected_restaurant["business_id"]

with REVIEW_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        review = json.loads(line)

        if review["business_id"] == business_id:
            reviews.append(review)

            if len(reviews) == 5:
                break

print(f"\nFOUND {len(reviews)} REVIEWS\n")

for i, review in enumerate(reviews, start=1):
    print(f"{'=' * 60}")
    print(f"REVIEW {i}")
    print(f"Stars: {review['stars']}")
    print(f"Date: {review['date']}")
    print(f"Text:\n{review['text']}\n")

SELECTED RESTAURANT
Name: O'Briens Irish Pub
Business ID: Ucl9Vo5lwrUmYbV8Dv8X5g
Categories: Bars, Pubs, Nightlife, Arts & Entertainment, Irish, Restaurants, Music Venues
Rating: 4.0
Review count: 108

FOUND 5 REVIEWS

REVIEW 1
Stars: 3.0
Date: 2013-12-08 23:11:18
Text:
I must say that this location is much better than the old location they used to occupy further south in Carrollwood.  There's always plenty of parking and it's a pretty convenient location.  My usual adventures here have been for their annual St. Patrick's Day blowout, which is under a huge tent outdoors, and to watch Gator football games as they are the north Tampa watch location for the Tampa Gator Club.  I've always been met with attentive servers and bartenders every time I have visited.  The food is your average run of the mill Irish bar food, but I think the main attraction is the atmosphere this place provides.

REVIEW 2
Stars: 1.0
Date: 2014-06-30 13:51:09
Text:
I was a huge fan of the old O'Briens with the dive

## Part 2: Tampa restaurant business subset

In [7]:
import json

TAMPA_BUSINESS_FILE = DATA_DIR / "tampa_restaurants.json"

restaurant_count = 0

with BUSINESS_FILE.open("r", encoding="utf-8") as source:
    with TAMPA_BUSINESS_FILE.open("w", encoding="utf-8") as destination:

        for line in source:
            business = json.loads(line)

            categories = [
                category.strip().lower()
                for category in (business.get("categories") or "").split(",")
            ]

            if (
                business.get("city", "").strip().lower() == "tampa"
                and "restaurants" in categories
            ):
                destination.write(json.dumps(business) + "\n")
                restaurant_count += 1

print(f"Tampa restaurants saved: {restaurant_count:,}")
print(f"Output file: {TAMPA_BUSINESS_FILE}")
print(f"File size: {TAMPA_BUSINESS_FILE.stat().st_size / (1024**2):.2f} MB")

Tampa restaurants saved: 2,967
Output file: ..\Yelp_JSON\tampa_restaurants.json
File size: 3.47 MB


#### 2a


In [8]:
TAMPA_REVIEW_FILE = DATA_DIR / "tampa_restaurant_reviews.jsonl"

# Load the IDs of our Tampa restaurants into a set.
tampa_business_ids = set()

with TAMPA_BUSINESS_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        business = json.loads(line)
        tampa_business_ids.add(business["business_id"])

print(f"Tampa business IDs loaded: {len(tampa_business_ids):,}")

# Stream through the original Yelp review dataset.
total_reviews = 0
tampa_reviews = 0

with REVIEW_FILE.open("r", encoding="utf-8") as source:
    with TAMPA_REVIEW_FILE.open("w", encoding="utf-8") as destination:

        for line in source:
            review = json.loads(line)
            total_reviews += 1

            if review["business_id"] in tampa_business_ids:
                destination.write(line)
                tampa_reviews += 1

print(f"\nTotal Yelp reviews scanned: {total_reviews:,}")
print(f"Tampa restaurant reviews saved: {tampa_reviews:,}")
print(f"Output file: {TAMPA_REVIEW_FILE}")
print(f"File size: {TAMPA_REVIEW_FILE.stat().st_size / (1024**2):.2f} MB")

Tampa business IDs loaded: 2,967

Total Yelp reviews scanned: 6,990,280
Tampa restaurant reviews saved: 303,691
Output file: ..\Yelp_JSON\tampa_restaurant_reviews.jsonl
File size: 214.43 MB


#### 2b



In [9]:
# Verify that every Tampa review references a valid Tampa restaurant.

total_reviews = 0
valid_reviews = 0
invalid_reviews = 0
restaurants_with_reviews = set()

with TAMPA_REVIEW_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        review = json.loads(line)
        total_reviews += 1

        business_id = review["business_id"]

        if business_id in tampa_business_ids:
            valid_reviews += 1
            restaurants_with_reviews.add(business_id)
        else:
            invalid_reviews += 1

restaurants_without_reviews = tampa_business_ids - restaurants_with_reviews

print("TAMPA DATA INTEGRITY CHECK")
print("-" * 40)
print(f"Total reviews checked:        {total_reviews:,}")
print(f"Valid reviews:                {valid_reviews:,}")
print(f"Invalid reviews:              {invalid_reviews:,}")
print(f"Restaurants with reviews:     {len(restaurants_with_reviews):,}")
print(f"Restaurants without reviews:  {len(restaurants_without_reviews):,}")

if invalid_reviews == 0:
    print("\nPASS: All reviews reference valid Tampa restaurants.")
else:
    print("\nFAIL: Some reviews reference unknown businesses.")

TAMPA DATA INTEGRITY CHECK
----------------------------------------
Total reviews checked:        303,691
Valid reviews:                303,691
Invalid reviews:              0
Restaurants with reviews:     2,967
Restaurants without reviews:  0

PASS: All reviews reference valid Tampa restaurants.


## Part 3: 
I

In [ ]:
import os
from nbconvert import PDFExporter
import nbformat

NOTEBOOK_NAME = "tester_Ben_Prishtina.ipynb"
NOTEBOOK_PATH = NOTEBOOK_NAME

# Read the notebook as UTF-8
with open(NOTEBOOK_PATH, encoding="utf-8") as f:
    nb_node = nbformat.read(f, as_version=4)

# Set PDF document title
nb_node.metadata["title"] = "Appendix: CAP 6938 - Final Project Tester"

pdf_exporter = PDFExporter()
pdf_data, resources = pdf_exporter.from_notebook_node(nb_node)

NOTEBOOK_PDF = NOTEBOOK_PATH.rsplit('.', 1)[0] + '.pdf'
with open(NOTEBOOK_PDF, "wb") as f:
    f.write(pdf_data)
    
print(f"PDF CREATED: {NOTEBOOK_PDF}")